# AtB Data Integration

This notebook validates the pipeline end to end while keeping the SMT adapter path close to the current implementation for verification.

In [ ]:
# %reset -f
%load_ext autoreload
%autoreload complete --log


In [ ]:
from smtgraphformer import *
from smtgraphformer.dataIntegration import atbBuilder
from smtgraphformer.contextEncoding import configCE
from smtgraphformer.modelAdapters import tfmTripLevelSMT
from smtgraphformer.smtGraphFormer import *

setDisplayOptions()
sr = setReproducibility(17711)


In [ ]:
root = Path("../data")
fp = root.joinpath("atbData-May2024-stopLevel-[fPM.eST.eLU.eDW].pkl")
assert fp.exists(), "!!!"

# set date ranges for train/valid/test splits
dsRanges = {
    "train": ("2024-05-01", "2024-05-21"),
    "valid": ("2024-05-22", "2024-05-26"),
    "test": ("2024-05-27", "2024-05-31"),
}


## Shared Artefacts
Build the canonical stop-level dataset, chronological split plan, stop attributes, line attributes, and the train-only transform bundle before deriving any model-specific view.

In [ ]:
builder = atbBuilder(verbose=False)
dataBCS = builder.buildCanonicalStops(fp)
ds_splits = builder.buildSplitPlan(dataBCS, dsRanges["train"], dsRanges["valid"], dsRanges["test"])

bundle = builder.fitTransformBundle(dataBCS, ds_splits)
dataFTB = builder.applyBundle(dataBCS, bundle, ds_splits)

sattrs = builder.extractStopAttributes(bundle)
lattrs = builder.extractLineAttributes()


In [ ]:
trip_counts = ds_splits.groupby("$split", observed=True).size().rename("n_trips").to_dict()
stop_counts = dataFTB.groupby("$split", observed=True).size().rename("n_stops").to_dict()
print(f"{trip_counts=}")
print(f"{stop_counts=}")

lfpreview = lambda tag, df: print(f"{tag}:\n{df.iloc[3].to_dict()}\n")
lfpreview("dataFTB", dataFTB)


In [ ]:
b_summary = {
    "stopids": len(bundle.mapStopIDs.maps),
    "categories": list(bundle.categoryMaps.maps),
    "continuous": list(bundle.continuousStats.means),
    "targets": list(bundle.targetTransforms),
}
print(b_summary)


## SMT Adapter Verification
Mirror the existing trip-level transformation shape closely so the new adapter remains easy to diff against the V1 path.

In [ ]:
required = [
    "dateTrip",
    "$split",
    "StopIdentifier",
    "StopSequence",
    "tfm.StopScheduledArrival",
    *[f"tfm.{c}" for c in builder.scContextCat],
    *[f"tfm.{c}" for c in builder.scContextCont],
    *[f"tfm.{c}.sin" for c in builder.scCyclic],
    *[f"tfm.{c}.cos" for c in builder.scCyclic],
    *[f"tfm.{c}" for c in builder.scTarget],
]
missing = [c for c in required if c not in dataFTB.columns]
assert not missing, "!!!"

tripsSMT = tfmTripLevelSMT(dataFTB)
n_unique = dataFTB["dateTrip"].nunique()
assert len(tripsSMT) == n_unique, "!!!"
assert sanityCheck(tripsSMT), "!!!"


In [ ]:
# sanity check: compare a sample trip in tripsSMT with the corresponding rows in dataFTB
t_sample = tripsSMT.iloc[3]
t_sample_ids = t_sample["StopIdentifier"].split(",")

t_source = dataFTB.loc[dataFTB["dateTrip"] == t_sample["dateTrip"]].copy()
t_source["$order"] = t_source["StopSequence"].astype(int)
t_source = t_source.sort_values("$order")
t_source_ids = t_source["StopIdentifier"].astype(str).tolist()

assert t_sample_ids == t_source_ids, "!!!"  # stop identifiers should match
assert t_sample["$split"] == t_source["$split"].iloc[0], "!!!"  # split should match


## Graph and Dataloader Smoke Test
Keep the downstream SMT path close to the current training setup by reusing relational matrices, graph autoencoder training, and the same batch structure expected by the model.

In [ ]:
mobilityPatterns = {
    (t, r): builder.aggTrainMobilityPattern(dataBCS, bundle, target=t, resolution=r)
    for t in ["boarding", "alighting"]
    for r in ["hourly", "daily"]
}

for (t, r), mp in mobilityPatterns.items():
    print(f"historical mobility pattern ({t}, {r}): {mp.shape}")


In [ ]:
BRM = buildRelationalMatrices()

stoprms = []
stoprms.append(BRM.stopDistance(sattrs))
stoprms.append(BRM.attributeSimilarity(sattrs, features=builder.scGraph))
for mp in mobilityPatterns.values():
    stoprms.append(BRM.mobilityPattern(mp))

ref = stoprms[0].shape
assert all(rm.shape == ref for rm in stoprms), "relational matrices have inconsistent shapes!"


In [ ]:
# print summary statistics
tags = [
    "stopDistance",
    "attributeSimilarity",
    *[f"mobilityPattern ({t}/{r})" for (t, r) in mobilityPatterns.keys()],
]
print(f"x{len(stoprms)} relational matrices > shape: {ref} > {len(sattrs)} stops + special tokens")
for t, rm in zip(tags, stoprms):
    print(f" - {t:<35}: min={rm.min()}, max={rm.max():.1f}, mean={rm.mean():.4f}, std={rm.std():.4f}")

# stack all matrices; add dummy batch dimension > [1, n_matrices, n_nodes, n_nodes]
t_stoprms = torch.tensor(np.stack(stoprms), dtype=torch.float32).unsqueeze(0)
print(f" > stacked matrices tensor: {t_stoprms.shape}")


In [ ]:
cfgModel = SMTConfig(
    embed_dim=64,
    batch_size=128,
    n_head=1,
    n_layer=1,
    n_expert=3,
    train_epochs=1,
    eval_interval=1,
    model_dir="../models",
    model_name="smtM24-Dummy",
    early_stopping=None,
)

dls = SMTDataloader(tripsSMT, cfgModel, bundle=bundle, sattrs=sattrs)
cfgModel.ctx_length = dls.ctx_length
cfgModel.vocab_size = dls.vocab_size

cfgContext = configCE(
    lc_cats=dls.ce_lccats,
    conts=dls.ce_nconts,
    output=cfgModel.trip_context_dim,  # type:ignore
)


In [ ]:
gaeModel, gaeEmbeddings = trainGAEModelEpochs(t_stoprms, cfgModel, n_epochs=128)
model = SMTGraphFormer(cfgModel, cfgContext, gaeEmbeddings, dls.stopFeatures)
print(f"model initialised with {sum(p.numel() for p in model.parameters()):,} parameters")


In [ ]:
model.dryrun(dls)


In [ ]:
log = trainSMTModelEpochs(m=model, dls=dls, cfg=cfgModel, save_model=False, final_eval=True)
pd.DataFrame(log).describe().T.round(4)


In [ ]:
plotTrainingHistory(log)


## Evaluation and Persistence
Use the shared evaluator in real-space units and save small verification artefacts that make behaviour comparisons easier during review.

In [ ]:
l_metrics = []
dfs_comparison = {}
for split in ["train", "valid", "test"]:
    x1_metrics, x1_comparison = smtFinalEvaluation(model, dls, bundle, split, teacher_forcing=True)
    x1_metrics.insert(0, "$split", split)
    l_metrics.append(x1_metrics)
    dfs_comparison[split] = x1_comparison

metrics = pd.concat(l_metrics, ignore_index=True)
print(metrics.tail(4))


In [ ]:
dst = Path("../data")

# save data builder artefacts
saved = builder.saveArtefacts(dst, tag=None)

# save trip-level SMT data
tripsSMT.to_pickle(f"{dst}/AtB-tfmTripLevelSMT.pkl")
fp = f"{dst}/AtB-tfmTripLevelSMT-Samplex5.json"
tripsSMT.iloc[[1, 17, 177, 1771, 17711]].to_json(fp, orient="records", indent=2)

# save relational matrices
fp = f"{dst}/AtB-buildRelationalMatrices.npy"
np.save(fp, t_stoprms.squeeze(0).numpy())


### end